# Agentic Search Tools

- Tools are frequently used in agentic search
- Flow - A agent would receive a prompt and produce an answer, agent decides whether it needs to call a search tool for proving the answer. The information found with the search tool is returned to the agent.
- Inside the search tool - 
    1. Understand query sent and divide into sub parts
    2. For each sub-query, the search tool should find the best source (retrieve) 
    3. Score retrieved results and fitler out only the relavant information needed for the sub-query
- Eg: Tavily agentic search tool

#### _Tavily Agentic Search Tool:_

Import tavily client and connect to it with API key:

    import os
    from tavily import TavilyClient

    client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

Run search on tavily search tool:

    # run search
    result = client.search("What is in Nvidia's new Blackwell GPU?",
                        include_answer=True)

    # print the answer
    result["answer"]


#### _Regular Search Example:_

Example Search Query:

    city = "San Francisco"

    query = f"""
        what is the current weather in {city}?
        Should I travel there today?
        "weather.com"
    """

Import duck duck go search and create an instance:

    import requests
    from bs4 import BeautifulSoup
    from duckduckgo_search import DDGS
    import re

    ddg = DDGS()

Search query using ddg and return links that might provide to the answer of the query:

    def search(query, max_results=6):
        try:
            results = ddg.text(query, max_results=max_results)
            return [i["href"] for i in results]
        except Exception as e:
            print(f"returning previous results due to exception reaching ddg.")
            results = [ # cover case where DDG rate limits due to high deeplearning.ai volume
                "https://weather.com/weather/today/l/USCA0987:1:US",
                "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
            ]
            return results  


    for i in search(query):
        print(i)

Scrape data from links using Beautiful soup to parse html data and handle error (not url link or cannot fetch data from webpage):

    def scrape_weather_info(url):
        """Scrape content from the given URL"""
        if not url:
            return "Weather information could not be found."
        
        # fetch data
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            return "Failed to retrieve the webpage."

        # parse result
        soup = BeautifulSoup(response.text, 'html.parser')
        return soup


Scrape first url link found using duck duck go on the query:

    # use DuckDuckGo to find websites and take the first result
    url = search(query)[0]

    # scrape first wesbsite
    soup = scrape_weather_info(url)

    print(f"Website: {url}\n\n")
    print(str(soup.body)[:50000]) # limit long outputs

Find all texts in h1, h2, h3, p and remove spaces to append into weather data list:

    # extract text
    weather_data = []
    for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
        text = tag.get_text(" ", strip=True)
        weather_data.append(text)

    # combine all elements into a single string
    weather_data = "\n".join(weather_data)

    # remove all spaces from the combined text
    weather_data = re.sub(r'\s+', ' ', weather_data)
        
    print(f"Website: {url}\n\n")
    print(weather_data)

#### _Agentic Search Example:_

Run Example Query on Taviliy search tool client:

    # run search
    result = client.search(query, max_results=1)

    # print first result
    data = result["results"][0]["content"]

    print(data)

=> If we specify include answer as True in the agentic search tool, then it will show the answer as text.

=> Since we did not specify to include answer in this search example and printed the first result, it will send result as json 

Parse Json data and format the data:

    import json
    from pygments import highlight, lexers, formatters

    # parse JSON
    parsed_json = json.loads(data.replace("'", '"'))

    # pretty print JSON with syntax highlighting
    formatted_json = json.dumps(parsed_json, indent=4)
    colorful_json = highlight(formatted_json,
                            lexers.JsonLexer(),
                            formatters.TerminalFormatter())

    print(colorful_json)
